In [1]:
!sudo apt install flex
!flex -o lex.yy.c lexer.l
!sudo apt install python3-dev
!gcc -fPIC $(python3-config --includes) -Wall -O2 -c lex.yy.c -o lex.yy.o
!gcc -fPIC $(python3-config --includes) -Wall -O2 -c wrapper.c -o wrapper.o
!gcc -shared $(python3-config --ldflags) lex.yy.o wrapper.o -o sql_tokenizer.so

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libfl-dev libfl2
Suggested packages:
  bison flex-doc
The following NEW packages will be installed:
  flex libfl-dev libfl2
0 upgraded, 3 newly installed, 0 to remove and 35 not upgraded.
Need to get 324 kB of archives.
After this operation, 1,148 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 flex amd64 2.6.4-8build2 [307 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl2 amd64 2.6.4-8build2 [10.7 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl-dev amd64 2.6.4-8build2 [6,236 B]
Fetched 324 kB in 1s (479 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 3.)
debconf: falling back to frontend

In [2]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier, RandomForestClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, RadiusNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier, Perceptron, PassiveAggressiveClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC, LinearSVC, NuSVC
from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
from sklearn.neighbors import NearestCentroid
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfTransformer,
    TfidfVectorizer,
)
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import FeatureUnion, Pipeline, make_pipeline, make_union
from sklearn.svm import SVC
from sql_tokenizer import tokenize
from xgboost import XGBClassifier
import time
from sklearn.linear_model import SGDClassifier


In [3]:
import html
import re


def decode_encodings(text: str) -> str:
    assert isinstance(text, str)
    try:
        text = text.replace("\\", "\\\\")
        text = text.encode("utf-8").decode("unicode_escape")
    except Exception as e:
        return ""
    text = re.sub(
        r"%([0-9A-Fa-f]{2})", lambda m: bytes.fromhex(m.group(1)).decode("latin1"), text
    )
    text = re.sub(r"[Uu]\+([0-9A-Fa-f]{4,6})", lambda m: chr(int(m.group(1), 16)), text)
    text = html.unescape(text)
    return text

In [4]:
class BoCVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.vectorizer = TfidfVectorizer(analyzer="char")

    def fit(self, X, y=None):
        return self.vectorizer.fit(X)

    def transform(self, X):
        return self.vectorizer.transform(X)


class Preprocessor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        lower = X.str.lower()
        return lower.apply(decode_encodings)

feature_sets = {
    "BoC only": make_pipeline(Preprocessor(), BoCVectorizer()),
    "Grammer aware TF-IDF": make_pipeline(
        Preprocessor(),
        CountVectorizer(
            tokenizer=tokenize,
            preprocessor=lambda x: x,
        ),
        TfidfTransformer(),
    ),
    "Grammer aware TF-IDF with n-gram": make_pipeline(
        Preprocessor(),
        CountVectorizer(
            tokenizer=tokenize,
            preprocessor=lambda x: x,
            ngram_range=(1, 3),
        ),
        TfidfTransformer(),
    ),
    "Combined features": make_pipeline(
        Preprocessor(),
        make_union(
            BoCVectorizer(),
            TfidfVectorizer(analyzer="word"),
            TfidfVectorizer(analyzer="word", ngram_range=(1, 3)),
        ),
    ),
    "Grammer aware Combined features": make_pipeline(
        Preprocessor(),
        make_union(
            BoCVectorizer(),
            make_pipeline(
                Preprocessor(),
                CountVectorizer(
                    tokenizer=tokenize,
                    preprocessor=lambda x: x,
                    ngram_range=(1, 3),
                ),
                TfidfTransformer(),
            ),
            make_pipeline(
                Preprocessor(),
                CountVectorizer(
                    tokenizer=tokenize,
                    preprocessor=lambda x: x,
                ),
                TfidfTransformer(),
            ),
        ),
    ),
}
classifiers = {
    "sqlidps": make_pipeline(RandomForestClassifier(n_estimators=200)),
    "proposed-ensemble": make_pipeline(
        VotingClassifier(
            estimators=[
                ("nb", MultinomialNB()),
                ("svm", SVC(kernel="linear", probability=True)),
                ("xgb", XGBClassifier(eval_metric="logloss")),
            ],
            voting="soft",
        )
    ),
    "AdaBoost": make_pipeline(StandardScaler(), AdaBoostClassifier()),
    "Bagging": make_pipeline(StandardScaler(), BaggingClassifier()),
    "DecisionTree": make_pipeline(StandardScaler(), DecisionTreeClassifier()),
    "ExtraTrees": make_pipeline(StandardScaler(), ExtraTreesClassifier()),
    "KNeighbors": make_pipeline(
        StandardScaler(), KNeighborsClassifier()
    ),
    "LinearSVC": make_pipeline(
        StandardScaler(), SVC(kernel="linear",  probability=True)
    ),
    "LogisticRegression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000)
    ),
    "MLP": make_pipeline(
        StandardScaler(), MLPClassifier(max_iter=1000)
    ),
    "MultinomialNB": make_pipeline(MultinomialNB()),
    "NearestCentroid": make_pipeline(StandardScaler(), NearestCentroid()),
    "NuSVC": make_pipeline(StandardScaler(), NuSVC()),
    "OneVsOne": make_pipeline(StandardScaler(), OneVsOneClassifier(LinearSVC())),
    "OneVsRest": make_pipeline(StandardScaler(), OneVsRestClassifier(LinearSVC())),
    "PassiveAggressive": make_pipeline(
        StandardScaler(), PassiveAggressiveClassifier(max_iter=1000)
    ),
    "Perceptron": make_pipeline(StandardScaler(), Perceptron(max_iter=1000)),
    "RidgeClassifier": make_pipeline(StandardScaler(), RidgeClassifier()),
    "SGDClassifier": make_pipeline(
        StandardScaler(), SGDClassifier(max_iter=1000, tol=1e-3)
    ),
    "SVC-GC": make_pipeline(
        StandardScaler(), SVC(kernel="poly",  probability=True)
    ),
    "SVM_RBF": make_pipeline(StandardScaler(), SVC(kernel="rbf",  probability=True)),
    "XGBoost": make_pipeline(
        StandardScaler(), XGBClassifier(eval_metric="mlogloss")
    ),
}

In [5]:
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, precision_recall_fscore_support)
from sklearn.model_selection import cross_val_score, cross_val_predict

In [ ]:


def train():
    np.set_printoptions(precision=5)
    pd.set_option("display.float_format", "{:.5f}".format)
    train_df = pd.read_csv("train.csv")
    test_df = pd.read_csv("test.csv")
    X_train, X_test, y_train, y_test = (
        train_df["Query"],
        test_df["Query"],
        train_df["Label"],
        test_df["Label"],
    )
    ncv = 5
    feature_cache = {}
    results = []
    cv = StratifiedKFold(n_splits=ncv, shuffle=True, random_state=42)
    for f_name, f_pipe in feature_sets.items():
      # caching feature head
      if f_name not in feature_cache:
          feature_cache[f_name] = f_pipe.fit_transform(X_train, y_train)
      X_feat = feature_cache[f_name]

      for c_name, clf in classifiers.items():
          y_true = []
          y_pred = []
          y_proba = []

          total_train_time = 0.0
          total_infer_time = 0.0
          for train_idx, test_idx in cv.split(X_feat, y_train):
              X_tr, X_te = X_feat[train_idx], X_feat[test_idx]
              y_tr, y_te = y_train[train_idx], y_train[test_idx]
              if hasattr(X_tr, "toarray"):
                  X_tr = X_tr.toarray()
                  X_te = X_te.toarray()

              start_train = time.perf_counter()
              clf.fit(X_tr, y_tr)
              end_train = time.perf_counter()

              start_infer = time.perf_counter()
              preds = clf.predict(X_te)
              end_infer = time.perf_counter()

              y_true.extend(y_te.values)
              y_pred.extend(preds)
              if hasattr(clf, "predict_proba"):
                  y_proba.extend(clf.predict_proba(X_te)[:, 1])
              else:
                  y_proba = None

              total_train_time += end_train - start_train
              total_infer_time += end_infer - start_infer
          acc = accuracy_score(y_true, y_pred)
          precision, recall, f1, _ = precision_recall_fscore_support(
              y_true, y_pred, average='weighted', zero_division=0
          )
          auc = roc_auc_score(y_true, y_proba) if y_proba else float('nan')
          avg_train_time = total_train_time / ncv
          avg_infer_time = total_infer_time / len(y_true)
          print(
              f"{f_name} + {c_name} → Acc: {acc:.4f}, Prec: {precision:.4f}, "
              f"Recall: {recall:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}, "
              f"TrainTime: {avg_train_time:.2f}s, InferTime: {avg_infer_time*1000:.5f}ms"
          )
          results.append({
              "feature": f_name,
              "classifier": c_name,
              "accuracy": acc,
              "precision": precision,
              "recall": recall,
              "f1": f1,
              "auc": auc,
              "avg_train_time": avg_train_time,
              "avg_infer_time": avg_infer_time
          })
    return results
if __name__ == "__main__":
    results = train()
    df = pd.DataFrame(results)
    df.to_csv("results.csv", index=False)

BoC only + sqlidps → Acc: 0.9972, Prec: 0.9972, Recall: 0.9972, F1: 0.9972, AUC: 0.9999, TrainTime: 22.18s, InferTime: 0.02459ms
BoC only + proposed-ensemble → Acc: 0.9925, Prec: 0.9925, Recall: 0.9925, F1: 0.9925, AUC: 0.9994, TrainTime: 63.84s, InferTime: 0.12908ms
BoC only + AdaBoost → Acc: 0.9761, Prec: 0.9761, Recall: 0.9761, F1: 0.9761, AUC: 0.9957, TrainTime: 9.90s, InferTime: 0.00952ms
BoC only + Bagging → Acc: 0.9928, Prec: 0.9928, Recall: 0.9928, F1: 0.9928, AUC: 0.9978, TrainTime: 23.18s, InferTime: 0.00511ms
BoC only + DecisionTree → Acc: 0.9912, Prec: 0.9912, Recall: 0.9912, F1: 0.9912, AUC: 0.9910, TrainTime: 4.21s, InferTime: 0.00094ms
BoC only + ExtraTrees → Acc: 0.9979, Prec: 0.9979, Recall: 0.9979, F1: 0.9979, AUC: 1.0000, TrainTime: 3.29s, InferTime: 0.01571ms
BoC only + KNeighbors → Acc: 0.9936, Prec: 0.9936, Recall: 0.9936, F1: 0.9936, AUC: 0.9986, TrainTime: 0.06s, InferTime: 0.42631ms
BoC only + LinearSVC → Acc: 0.9856, Prec: 0.9856, Recall: 0.9856, F1: 0.9856, A

/usr/local/lib/python3.11/dist-packages/sklearn/neighbors/_nearest_centroid.py:244: UserWarning: self.within_class_std_dev_ has at least 1 zero standard deviation.Inputs within the same classes for at least 1 feature are identical.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/neighbors/_nearest_centroid.py:244: UserWarning: self.within_class_std_dev_ has at least 1 zero standard deviation.Inputs within the same classes for at least 1 feature are identical.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/neighbors/_nearest_centroid.py:244: UserWarning: self.within_class_std_dev_ has at least 1 zero standard deviation.Inputs within the same classes for at least 1 feature are identical.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/neighbors/_nearest_centroid.py:244: UserWarning: self.within_class_std_dev_ has at least 1 zero standard deviation.Inputs within the same classes for at least 1 feature are identical.
  warnings.warn(
/usr

BoC only + NearestCentroid → Acc: 0.8925, Prec: 0.8922, Recall: 0.8925, F1: 0.8922, AUC: 0.9554, TrainTime: 0.11s, InferTime: 0.00096ms
